# Module 9: Frontiers — Offline RL Basics

## Learning Objectives
By completing this notebook, you will:
1. Distinguish offline RL from online RL and explain when each is appropriate
2. Demonstrate the distribution shift problem: why naive Q-learning diverges on fixed datasets
3. Implement a conservative Q-learning (CQL-lite) approach that penalizes out-of-distribution actions
4. Compare naive and conservative offline training on a GridWorld

## Prerequisites
- Module 3: Q-learning update rule
- Module 5: Experience replay and replay buffers
- Module 8: Tabular models

## Estimated Time: 15 minutes

---

## Offline RL: Learning Without a Simulator

All RL algorithms covered so far require **online interaction**: the agent acts, observes, and learns in a loop. Many real-world problems make online interaction expensive or dangerous:
- **Healthcare**: you cannot randomize treatment while learning
- **Autonomous driving**: mistakes during exploration kill people
- **Finance**: bad policies lose real money before you learn they're bad

**Offline RL** (also called batch RL) learns from a **fixed dataset** of transitions $(s, a, r, s')$ collected by some *behavior policy* $\mu$ — no environment interaction at all. The challenge: the learned policy may want to take actions that were never (or rarely) tried in the dataset. Q-values for those actions are extrapolated — and extrapolation in neural networks is notoriously unreliable.

This is the **distribution shift** problem: the deployment distribution $d^{\pi}$ differs from the data distribution $d^{\mu}$.

In [ ]:
learning_objectives([
    "Module 3: Q-learning update rule",
    "Module 5: Experience replay and replay buffers",
    "Module 8: Tabular models",
    "## Offline RL: Learning Without a Simulator",
    "**Healthcare**: you cannot randomize treatment while learning",
    "**Autonomous driving**: mistakes during exploration kill people",
    "**Finance**: bad policies lose real money before you learn they're bad",
    "(also called batch RL) learns from a **fixed dataset** of transitions $(s, a, r, s')$ collected by some *behavior policy* $\mu$ — no environment interaction at all. The challenge: the learned policy may want to take actions that were never (or rarely) tried in the dataset. Q-values for those actions are extrapolated — and extrapolation in neural networks is notoriously unreliable.",
    "problem: the deployment distribution $d^{\pi}$ differs from the data distribution $d^{\mu}$."
])

In [ ]:
import sys; sys.path.insert(0, '../../../../..')
from resources.notebook_style import apply_course_theme, learning_objectives, section_divider, callout, key_takeaways
from resources.graphics.plot_theme import apply_plot_theme

In [ ]:
apply_course_theme()
apply_plot_theme()

In [ ]:
# Purpose: Import dependencies
# Key Concept: Offline RL is tabular here, so no PyTorch needed

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from collections import defaultdict
import warnings

warnings.filterwarnings('ignore')

SEED = 42
np.random.seed(SEED)

print("Offline RL: Learning from a fixed dataset without environment interaction")

## 1. GridWorld Environment

A simple 5x5 GridWorld with walls. The goal is at the top-right corner. The offline dataset is collected by a partially-trained online Q-learning agent — it knows a decent but not optimal path.

In [ ]:
# Purpose: Define the GridWorld used for offline dataset collection and evaluation
# Key Concept: The same environment is used by the behavior policy (data) and evaluation (test)

class GridWorld:
    """
    5x5 GridWorld with walls.

    Actions: 0=up, 1=down, 2=left, 3=right
    Reward: +10 at goal, -1 per step, -5 at wall collision (bounce)
    """

    ACTIONS = [(-1, 0), (1, 0), (0, -1), (0, 1)]
    N_ACTIONS = 4

    def __init__(self, rows: int = 5, cols: int = 5):
        self.rows = rows
        self.cols = cols
        self.start = (rows - 1, 0)
        self.goal = (0, cols - 1)

        # Internal wall: vertical barrier at col 2 (rows 1-3)
        self.walls = {(1, 2), (2, 2), (3, 2)}
        self.state = self.start

    def reset(self) -> tuple:
        self.state = self.start
        return self.state

    def step(self, action: int) -> tuple:
        dr, dc = self.ACTIONS[action]
        r, c = self.state
        nr = max(0, min(self.rows - 1, r + dr))
        nc = max(0, min(self.cols - 1, c + dc))

        wall_hit = (nr, nc) in self.walls
        if wall_hit:
            nr, nc = r, c

        self.state = (nr, nc)
        done = (self.state == self.goal)
        reward = 10.0 if done else (-5.0 if wall_hit else -1.0)
        return self.state, reward, done

    @property
    def n_states(self) -> int:
        return self.rows * self.cols

    def state_idx(self, state: tuple) -> int:
        return state[0] * self.cols + state[1]

    def render_q(self, Q: np.ndarray, title: str = "", ax=None):
        """Render max Q values as a heatmap."""
        if ax is None:
            _, ax = plt.subplots(figsize=(5, 5))

        q_max = np.max(Q, axis=1).reshape(self.rows, self.cols)
        im = ax.imshow(q_max, cmap='RdYlGn', aspect='auto')

        for (r, c) in self.walls:
            ax.add_patch(mpatches.Rectangle((c - 0.5, r - 0.5), 1, 1,
                                            facecolor='black', alpha=0.9))
        ax.scatter([self.start[1]], [self.start[0]], s=200, c='blue', zorder=5, marker='o')
        ax.scatter([self.goal[1]], [self.goal[0]], s=200, c='gold', zorder=5, marker='*')

        ax.set_title(title, fontsize=11)
        ax.set_xticks(range(self.cols))
        ax.set_yticks(range(self.rows))
        return im


env = GridWorld()
print(f"GridWorld: {env.rows}x{env.cols} | States: {env.n_states} | Actions: {env.N_ACTIONS}")
print(f"Start: {env.start} | Goal: {env.goal} | Walls: {env.walls}")

## 2. Collecting an Offline Dataset

We first train an online Q-learning agent for 200 episodes (partially trained — not optimal). Then we record all its transitions as the offline dataset. The resulting dataset has high coverage of the states the behavior policy visits but sparse coverage elsewhere.

In [ ]:
# Purpose: Train an online Q-learning agent to collect the offline dataset
# Key Concept: The behavior policy quality determines dataset coverage — poor behavior = poor data

def train_online_agent(
    env: GridWorld,
    n_episodes: int = 200,
    alpha: float = 0.1,
    gamma: float = 0.95,
    epsilon: float = 0.3,  # high epsilon = more exploration in data
    seed: int = SEED
) -> tuple:
    """
    Train Q-learning agent and return final Q-table and trajectory.

    Returns
    -------
    tuple of (Q_table, list_of_episodes_rewards)
    """
    np.random.seed(seed)
    Q = np.zeros((env.n_states, env.N_ACTIONS))
    rewards_log = []

    for ep in range(n_episodes):
        state = env.reset()
        ep_r = 0.0
        done = False
        steps = 0

        while not done and steps < 100:
            s_idx = env.state_idx(state)
            if np.random.random() < epsilon:
                action = np.random.randint(env.N_ACTIONS)
            else:
                action = int(np.argmax(Q[s_idx]))

            next_state, reward, done = env.step(action)
            sp_idx = env.state_idx(next_state)

            td_target = reward + gamma * np.max(Q[sp_idx]) * (1 - int(done))
            Q[s_idx, action] += alpha * (td_target - Q[s_idx, action])

            state = next_state
            ep_r += reward
            steps += 1

        rewards_log.append(ep_r)

    return Q, rewards_log


def collect_dataset(
    env: GridWorld,
    Q_behavior: np.ndarray,
    n_episodes: int = 500,
    epsilon: float = 0.2,
    seed: int = SEED
) -> list:
    """
    Collect transitions using the behavior policy (epsilon-greedy over Q_behavior).

    Returns
    -------
    list of (state_idx, action, reward, next_state_idx, done) tuples
    """
    np.random.seed(seed + 1)
    dataset = []

    for _ in range(n_episodes):
        state = env.reset()
        done = False
        steps = 0

        while not done and steps < 100:
            s_idx = env.state_idx(state)
            if np.random.random() < epsilon:
                action = np.random.randint(env.N_ACTIONS)
            else:
                action = int(np.argmax(Q_behavior[s_idx]))

            next_state, reward, done = env.step(action)
            sp_idx = env.state_idx(next_state)
            dataset.append((s_idx, action, reward, sp_idx, done))
            state = next_state
            steps += 1

    return dataset


# Train behavior policy
print("Training behavior policy (online Q-learning)...")
Q_behavior, online_rewards = train_online_agent(env, n_episodes=200)
print(f"Behavior policy final 20-ep avg reward: {np.mean(online_rewards[-20:]):.1f}")

# Collect dataset
print("Collecting offline dataset...")
dataset = collect_dataset(env, Q_behavior, n_episodes=500)
print(f"Dataset size: {len(dataset)} transitions")

# Show state coverage
visited_states = set(s for s, a, r, sp, d in dataset)
print(f"State coverage: {len(visited_states)}/{env.n_states} states visited")

## 3. Naive Offline Q-Learning (Distribution Shift)

We apply standard Q-learning updates to the fixed dataset. The problem: whenever the learned policy selects an action not in the dataset, the Q-value for that action is unconstrained — it can grow arbitrarily large, leading the policy to take increasingly bad out-of-distribution actions.

In [ ]:
# Purpose: Implement naive offline Q-learning
# Key Concept: Bootstrapping with unconstrained Q-values causes overestimation and divergence

def offline_q_learning(
    dataset: list,
    n_states: int,
    n_actions: int,
    n_epochs: int = 50,
    alpha: float = 0.1,
    gamma: float = 0.95,
    batch_size: int = 32,
    seed: int = SEED
) -> tuple:
    """
    Naive offline Q-learning: standard Q-updates on the fixed dataset.

    Returns
    -------
    tuple of (Q_table, q_max_history)
        Q_table: learned Q values
        q_max_history: max Q value per epoch (divergence indicator)
    """
    np.random.seed(seed)
    Q = np.zeros((n_states, n_actions))
    dataset_arr = np.array(dataset, dtype=object)
    q_max_history = []

    for epoch in range(n_epochs):
        # Sample a random mini-batch from the dataset
        indices = np.random.choice(len(dataset), size=min(batch_size, len(dataset)),
                                   replace=False)
        for idx in indices:
            s_idx, action, reward, sp_idx, done = dataset[idx]

            # Standard Q-learning target (bootstraps with max Q in next state)
            td_target = reward + gamma * np.max(Q[sp_idx]) * (1 - int(done))
            Q[s_idx, action] += alpha * (td_target - Q[s_idx, action])

        q_max_history.append(np.max(Q))

    return Q, q_max_history


print("Training naive offline Q-learning...")
Q_naive, q_max_naive = offline_q_learning(
    dataset, env.n_states, env.N_ACTIONS, n_epochs=100
)
print(f"Naive offline: final max Q = {q_max_naive[-1]:.2f}")

## 4. Conservative Q-Learning (CQL-lite)

CQL (Kumar et al., 2020) adds a penalty term that pushes Q-values down for out-of-distribution actions:

$$\mathcal{L}^{\text{CQL}}(Q) = \mathcal{L}^{\text{standard}}(Q) + \alpha \cdot \underbrace{\mathbb{E}_{s \sim \mathcal{D}}\left[\log \sum_a \exp Q(s, a) - Q(s, a^{\text{data}})\right]}_{\text{conservative penalty}}$$

The penalty term forces Q-values up for the data action and down for all others. Our CQL-lite approximates this as:
- **Increase** Q for the observed (s, a) action
- **Decrease** Q for all other actions in state s

This penalizes actions the behavior policy never took, preventing overestimation.

In [ ]:
# Purpose: Implement conservative offline Q-learning (CQL-lite)
# Key Concept: The conservative penalty prevents Q-values for unobserved (s,a) pairs from exploding

def cql_lite(
    dataset: list,
    n_states: int,
    n_actions: int,
    n_epochs: int = 100,
    alpha: float = 0.1,
    gamma: float = 0.95,
    cql_alpha: float = 0.5,  # Conservative penalty strength
    batch_size: int = 32,
    seed: int = SEED
) -> tuple:
    """
    Conservative offline Q-learning.

    For each transition (s, a, r, s', done):
    1. Standard Q-update for Q(s, a)
    2. Conservative penalty: decrease Q(s, a') for all a' != a

    Parameters
    ----------
    cql_alpha : float
        Strength of conservative penalty (higher = more conservative).

    Returns
    -------
    tuple of (Q_table, q_max_history)
    """
    np.random.seed(seed)
    Q = np.zeros((n_states, n_actions))
    q_max_history = []

    # Build a lookup: which actions were taken in each state?
    observed_actions = defaultdict(set)
    for s_idx, action, r, sp_idx, done in dataset:
        observed_actions[s_idx].add(action)

    for epoch in range(n_epochs):
        indices = np.random.choice(len(dataset), size=min(batch_size, len(dataset)),
                                   replace=False)
        for idx in indices:
            s_idx, action, reward, sp_idx, done = dataset[idx]

            # Step 1: Standard Q-learning update for the observed (s, a)
            td_target = reward + gamma * np.max(Q[sp_idx]) * (1 - int(done))
            Q[s_idx, action] += alpha * (td_target - Q[s_idx, action])

            # Step 2: Conservative penalty for out-of-distribution actions
            # Push down Q-values for actions never observed in state s_idx
            for a_other in range(n_actions):
                if a_other not in observed_actions[s_idx]:
                    Q[s_idx, a_other] -= cql_alpha * alpha * abs(Q[s_idx, a_other])

        q_max_history.append(np.max(Q))

    return Q, q_max_history


print("Training CQL-lite offline agent...")
Q_cql, q_max_cql = cql_lite(
    dataset, env.n_states, env.N_ACTIONS, n_epochs=100
)
print(f"CQL-lite: final max Q = {q_max_cql[-1]:.2f}")

## 5. Evaluating Offline-Trained Policies

In [ ]:
# Purpose: Evaluate all three policies in the real environment
# Key Concept: Offline training quality is only meaningful at deployment — test in the real env

def evaluate_policy(
    env: GridWorld,
    Q: np.ndarray,
    n_episodes: int = 100,
    max_steps: int = 50,
    seed: int = SEED + 99
) -> dict:
    """
    Greedily evaluate a Q-table policy.

    Returns
    -------
    dict with keys: 'mean_reward', 'success_rate', 'mean_steps'
    """
    np.random.seed(seed)
    rewards, successes, steps_list = [], [], []

    for _ in range(n_episodes):
        state = env.reset()
        ep_r = 0.0
        done = False
        steps = 0

        while not done and steps < max_steps:
            s_idx = env.state_idx(state)
            action = int(np.argmax(Q[s_idx]))
            state, r, done = env.step(action)
            ep_r += r
            steps += 1

        rewards.append(ep_r)
        successes.append(int(done and state == env.goal))
        steps_list.append(steps)

    return {
        'mean_reward': np.mean(rewards),
        'success_rate': np.mean(successes),
        'mean_steps': np.mean(steps_list)
    }


results = {
    'Behavior Policy (online)': evaluate_policy(env, Q_behavior),
    'Naive Offline Q-learning': evaluate_policy(env, Q_naive),
    'CQL-lite (offline)': evaluate_policy(env, Q_cql),
}

print("\nPolicy Evaluation Summary")
print("=" * 55)
print(f"{'Policy':<30} {'Mean Reward':>11} {'Success':>8} {'Steps':>7}")
print("-" * 55)
for name, r in results.items():
    print(f"{name:<30} {r['mean_reward']:>11.2f} {r['success_rate']:>8.2%} {r['mean_steps']:>7.1f}")

## 6. Visualizing the Distribution Shift Problem

In [ ]:
# Purpose: Visualize Q-value evolution and learned value functions
# Key Concept: Diverging max Q in naive offline is the hallmark of distribution shift

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# --- Plot 1: Max Q value over training epochs ---
ax = axes[0]
ax.plot(q_max_naive, color='crimson', linewidth=2, label='Naive offline')
ax.plot(q_max_cql, color='darkgreen', linewidth=2, label='CQL-lite')
ax.set_xlabel('Training Epoch', fontsize=12)
ax.set_ylabel('Max Q(s, a) across all (s, a)', fontsize=12)
ax.set_title('Q-Value Growth: Distribution Shift', fontsize=12)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)

# --- Plot 2: Naive offline Q-values ---
ax = axes[1]
im1 = env.render_q(Q_naive, title='Naive Offline Q-learning\nmax Q(s,a)', ax=ax)
plt.colorbar(im1, ax=ax)

# --- Plot 3: CQL-lite Q-values ---
ax = axes[2]
im2 = env.render_q(Q_cql, title='CQL-lite (Conservative)\nmax Q(s,a)', ax=ax)
plt.colorbar(im2, ax=ax)

plt.tight_layout()
plt.savefig('offline_rl_comparison.png', dpi=100, bbox_inches='tight')
plt.show()
print("Figure saved to offline_rl_comparison.png")

## 7. Dataset Coverage Visualization

In [ ]:
# Purpose: Show which (state, action) pairs are covered in the dataset
# Key Concept: Gaps in coverage are where distribution shift occurs

fig, axes = plt.subplots(1, 2, figsize=(10, 4))

# Count transitions per (state, action)
coverage = np.zeros((env.n_states, env.N_ACTIONS))
for s_idx, action, r, sp_idx, done in dataset:
    coverage[s_idx, action] += 1

# State visit frequency
state_visits = coverage.sum(axis=1).reshape(env.rows, env.cols)

ax = axes[0]
im = ax.imshow(state_visits, cmap='Blues', aspect='auto')
plt.colorbar(im, ax=ax, label='Number of visits')
for (r, c) in env.walls:
    ax.add_patch(mpatches.Rectangle((c - 0.5, r - 0.5), 1, 1, facecolor='black', alpha=0.9))
ax.scatter([env.start[1]], [env.start[0]], s=200, c='blue', zorder=5, marker='o')
ax.scatter([env.goal[1]], [env.goal[0]], s=200, c='gold', zorder=5, marker='*')
ax.set_title('Dataset: State Visit Frequency', fontsize=12)
ax.set_xlabel('Column')
ax.set_ylabel('Row')

# Action coverage per state
ax = axes[1]
action_coverage = (coverage > 0).sum(axis=1).reshape(env.rows, env.cols)
im2 = ax.imshow(action_coverage, cmap='YlOrRd', aspect='auto', vmin=0, vmax=env.N_ACTIONS)
plt.colorbar(im2, ax=ax, label='Actions covered (of 4)')
for (r, c) in env.walls:
    ax.add_patch(mpatches.Rectangle((c - 0.5, r - 0.5), 1, 1, facecolor='black', alpha=0.9))
ax.scatter([env.start[1]], [env.start[0]], s=200, c='blue', zorder=5, marker='o')
ax.scatter([env.goal[1]], [env.goal[0]], s=200, c='gold', zorder=5, marker='*')
ax.set_title('Dataset: Action Coverage per State\n(white = no data = distribution shift risk)', fontsize=11)
ax.set_xlabel('Column')
ax.set_ylabel('Row')

plt.tight_layout()
plt.savefig('dataset_coverage.png', dpi=100, bbox_inches='tight')
plt.show()
print("Figure saved to dataset_coverage.png")

## Exercise 9.1: CQL Penalty Strength

**Task:** The `cql_alpha` parameter controls how strongly we penalize out-of-distribution actions. Train CQL-lite with `cql_alpha` in `{0.0, 0.1, 0.5, 2.0}` for 100 epochs. Report final max Q value and success rate for each.

**Expected observation:** `cql_alpha=0.0` is identical to naive offline Q-learning. Very high `cql_alpha` over-penalizes and may hurt performance even for in-distribution actions.

<details>
<summary>Hint 1</summary>
Call `cql_lite(dataset, env.n_states, env.N_ACTIONS, n_epochs=100, cql_alpha=c)` in a loop.
</details>

In [ ]:
# YOUR CODE HERE
# ---------------
cql_alphas = [0.0, 0.1, 0.5, 2.0]
cql_alpha_results = {}  # Map cql_alpha -> {'max_q': float, 'success_rate': float}

# Fill cql_alpha_results
# Print a summary table

In [ ]:
# AUTO-GRADED TESTS - Do not modify
# ----------------------------------
def test_exercise_9_1():
    assert len(cql_alpha_results) == 4, \
        f"Expected 4 cql_alpha experiments, got {len(cql_alpha_results)}"
    for c in cql_alphas:
        assert c in cql_alpha_results, f"Missing result for cql_alpha={c}"
        d = cql_alpha_results[c]
        assert 'max_q' in d, f"Results for cql_alpha={c} must have 'max_q' key"
        assert 'success_rate' in d, f"Results for cql_alpha={c} must have 'success_rate' key"
        assert 0.0 <= d['success_rate'] <= 1.0, \
            f"success_rate must be in [0,1], got {d['success_rate']}"

    # cql_alpha=0.0 should behave like naive: higher max Q than cql_alpha=0.5
    q0 = cql_alpha_results[0.0]['max_q']
    q_mid = cql_alpha_results[0.5]['max_q']
    # Not strictly enforced due to randomness, but check the values are plausible
    assert q0 >= 0, f"max_q for alpha=0.0 should be non-negative, got {q0}"
    assert q_mid >= 0, f"max_q for alpha=0.5 should be non-negative, got {q_mid}"

    print("Exercise 9.1 passed! CQL penalty strength analysis complete.")

test_exercise_9_1()

## Exercise 9.2: Dataset Quality Impact

**Task:** Collect three datasets with different behavior policy qualities by varying the behavior policy's training length: `n_episodes_behavior` in `{50, 200, 500}`. For each dataset, train CQL-lite (cql_alpha=0.5) and evaluate the resulting policy. Report success rate.

**Expected observation:** Better behavior policy → better data coverage → better offline-trained policy.

<details>
<summary>Hint 1</summary>
Use `train_online_agent(env, n_episodes=n)` to get a behavior Q-table, then `collect_dataset(env, Q_beh, n_episodes=300)` for the dataset.
</details>

In [ ]:
# YOUR CODE HERE
# ---------------
behavior_qualities = [50, 200, 500]  # n_episodes to train behavior policy
quality_results = {}  # Map n_episodes -> success_rate

# Fill quality_results
# Print results

In [ ]:
# AUTO-GRADED TESTS - Do not modify
# ----------------------------------
def test_exercise_9_2():
    assert len(quality_results) == 3, \
        f"Expected 3 quality experiments, got {len(quality_results)}"
    for n in behavior_qualities:
        assert n in quality_results, f"Missing result for n_episodes={n}"
        sr = quality_results[n]
        assert isinstance(sr, (int, float)), \
            f"quality_results[{n}] should be a float success rate, got {type(sr)}"
        assert 0.0 <= sr <= 1.0, \
            f"Success rate must be in [0, 1], got {sr}"
    print("Exercise 9.2 passed! Dataset quality analysis complete.")

test_exercise_9_2()

In [ ]:
section_divider("Key Takeaways")

## Key Takeaways

1. **Offline RL decouples data collection from policy optimization** — critical for applications where online exploration is dangerous, expensive, or impossible.

2. **Distribution shift is the core problem**: bootstrapping with Q-values for (s, a) pairs absent from the dataset causes overestimation. The policy then selects these over-estimated actions, worsening the problem recursively.

3. **Conservative approaches (CQL, IQL, TD3+BC)** add a regularization term that penalizes Q-values for out-of-distribution actions. This keeps the learned policy close to the behavior policy's support.

4. **Dataset quality determines ceiling performance**: no offline algorithm can outperform a policy whose actions never appear in the data. High-quality, diverse offline data is the first requirement.

5. **The tradeoff**: too little conservatism → distribution shift divergence. Too much conservatism → policy is too close to (possibly suboptimal) behavior policy. CQL-alpha must be tuned.

## What's Next

`02_rl_trading_environment.ipynb` applies RL to sequential decision making in financial markets — building a custom Gymnasium environment for stock trading and training a Q-learning agent.

## Additional Resources

- Kumar et al. (2020): "Conservative Q-Learning for Offline Reinforcement Learning" — original CQL paper
- Levine et al. (2020): "Offline Reinforcement Learning: Tutorial, Review, and Perspectives"
- D4RL: https://github.com/rail-berkeley/d4rl — standard offline RL benchmark datasets

In [ ]:
key_takeaways([
    "Review the key concepts covered in this notebook",
    "Practice applying these techniques to your own data",
    "Connect this material to the companion guide and slides"
])